In [27]:
#defining a zero-lag ema
def zlema(series, period):
    ema1 = talib.EMA(series, period)
    ema2 = talib.EMA(ema1, period)
    return 2 * ema1 - ema2
#implementig vectorized operation to define crossovers 
def vectorized_crossover(series1, series2):
    return (series1 > series2) & (series1.shift(1) < series2.shift(1))
    
def vectorized_crossunder(series1, series2):
    return (series1 < series2) & (series1.shift(1) > series2.shift(1))
    
#defining a zero-lag macd, cond_buy is defined whenever macd line and signal line crosses under zero line
#simmetrically for cond_sell
def macd_impl(df):
    df['fast_period'] = zlema(df['Close'], 12)
    df['slow_period'] = zlema(df['Close'], 26)
    df['macd'] = df['fast_period'] - df['slow_period']
    df['signal'] = zlema(df['macd'], 9)
    df['hist'] = df['macd'] - df['signal']
    df['atr'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    
    cond_buy = (
    vectorized_crossover(df['macd'], df['signal']) & 
    (df['macd'] < 0)
    )
    
    cond_sell = (
    vectorized_crossunder(df['macd'], df['signal']) & 
    (df['macd'] > 0)
    )
    conditions = [cond_buy, cond_sell]
    choices    = [1, -1]
#Trade_Direction will translate cond_buy and cond_sell into 1 and -1
    df['Trade_Direction'] = np.select(conditions, choices, default=0)
    df['stop_loss'] = np.where(
        df['Trade_Direction'] > 0,
        df['Close'] - (df['atr'] * 2.5),
        df['Close'] + (df['atr'] * 2.5)
         )
    #df['Trade_Direction'] = df['Trade_Direction'].shift(-1)
    return df

In [28]:
#this function will aling the trade result with the trigger candle
def implement_trades(df, df_results):
    
    df['Trades'] = np.nan
    
    df.iloc[df_results['EntryBar'].values, df.columns.get_loc('Trades')] = df_results['PnL'].values
    
    df.loc[df['Trades'] > 0, 'Trades'] = 1  #PnL > 0 win trade
    df.loc[df['Trades'] < 0, 'Trades'] = 0 #PnL <0 loss trade

    return df

In [29]:
def macd_features(df):
    #macd velocity
    df['macd_slope'] = df['macd'].diff(3) / (df['atr'])
    #histogram impulse
    df['hist_delta'] = (df['hist'] - df['hist'].shift(1)) / df['atr']
    #zero line distance
    df['macd_z_score'] = (df['macd'] - df['macd'].rolling(200).mean()) / df['macd'].rolling(200).std()
    #since last cross
    s = np.sign(df['macd'] - df['macd'])
    df['bars_since_cross'] = s.groupby((s != s.shift()).cumsum()).cumcount()
    return df

In [30]:
def volume_features(df):
    
    df['vol_ma'] = df['Volume'].rolling(20).mean()
    df['rvol'] = df['Volume'] / df['vol_ma']
    df['vwap'] = (df['Volume'] * (df['High']+df['Low']+df['Close'])/3).cumsum() / df['Volume'].cumsum()
    df['dist_to_vwap'] = (df['Close'] - df['vwap']) / df['atr']
    
    return df

In [31]:
def price_features(df):

    df['price_slope'] = talib.LINEARREG_SLOPE(df['Close'], timeperiod = 20) / df['Close'] * 100

    don_upper = talib.MAX(df['High'], timeperiod = 20)
    don_lower = talib.MIN(df['Low'], timeperiod = 20)
    df['donchain_pos'] = (df['Close'] - don_lower) / (don_upper - don_lower)

    #candle size
    df['upper_wick'] = df['High'] - np.maximum(df['Close'], df['Open'])
    df['wick_ratio'] = df['upper_wick'] / (df['High'] - df['Low'])
    
    return df

In [32]:
def indicators_features(df):
    
    df['sar_dist'] = (df['Close'] - (talib.SAR(df['High'], df['Low']))) / df['atr']
    df['Rsi_9'] = talib.RSI(df['Close'], timeperiod = 9)
    df['Rsi_14'] = talib.RSI(df['Close'], timeperiod = 14)

    df['adx'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod = 14)    
    df['plus_di'] = talib.PLUS_DI(df['High'], df['Low'], df['Close'], timeperiod = 14)
    df['minus_di'] = talib.MINUS_DI(df['High'], df['Low'], df['Close'], timeperiod = 14)
    df['di_diff'] = df['plus_di'] - df['minus_di']
    
    return df

In [33]:
def ema_features(df):
    
    ema_50 = talib.EMA(df['Close'], 50)
    ema_200 = talib.EMA(df['Close'], 200)
    
    df['ema_50_slope'] = (talib.LINEARREG_SLOPE(ema_50, timeperiod = 50)) / df['Close'] *100
    df['ema_200_slope'] = (talib.LINEARREG_SLOPE(ema_200, timeperiod = 50)) / df['Close'] * 100
    
    df['ema_gap'] = (ema_50 - ema_200) / ema_200 * 100
    
    df['distance_from_ema_50'] = (df['Close'] - ema_50).abs() / df['Close']
    df['distance_from_ema_200'] = (df['Close'] - ema_200).abs() / df['Close']

    df['1h_ema'] = df['Close'].resample('1H').mean().reindex(df.index, method='ffill')
    df['htf_trend_diff'] = (df['Close'] - df['1h_ema']) / df['Close']

    return df

In [34]:
def get_ribbon_features(df, base_period = 10, step = 10, count = 6):

    ema_list = []
    for i in range(count):
        period = base_period + (i * step)
        col_name = f'ema_{period}'
        df[col_name] = talib.EMA(df['Close'], timeperiod=period)
        ema_list.append(col_name)
    
    df['ribbon_width'] = df[ema_list].std(axis=1) / df[ema_list].mean(axis=1)
    
    # 1 = Bullish fan, -1 = Bearish fan, 0 = Tangled/Range
    df['ribbon_direction'] = np.where((df[ema_list[0]] > df[ema_list[-1]]), 1, -1)
    
    return df

In [35]:
import pandas as pd
import talib
import numpy as np
from backtesting import Strategy, Backtest

In [36]:
df = pd.read_csv('btcusdt_spot_last_3years_italy.csv')
df.set_index('timestamp_rome', inplace=True)
df.sort_index(inplace=True)
df.index = pd.to_datetime(df.index, utc=True)

In [37]:
df

,open,high,low,close,volume,trades
timestamp_rome,,,,,,
2022-12-09 11:45:00+00:00,17234.57,17248.13,17233.73,17246.19,806.14043,15510
2022-12-09 11:50:00+00:00,17246.64,17257.28,17237.69,17255.22,1381.87009,24170
2022-12-09 11:55:00+00:00,17255.22,17255.99,17239.56,17241.44,933.02777,16893
2022-12-09 12:00:00+00:00,17241.16,17252.63,17241.16,17252.13,726.91979,16134
2022-12-09 12:05:00+00:00,17251.68,17252.89,17239.03,17246.01,719.44310,12937
...,...,...,...,...,...,...
2025-12-08 11:20:00+00:00,92150.92,92188.31,92123.24,92130.62,25.11894,9086
2025-12-08 11:25:00+00:00,92130.62,92130.62,92040.21,92040.21,12.72705,5801
2025-12-08 11:30:00+00:00,92040.21,92040.22,91851.08,91916.98,61.13074,19271


In [38]:
df.columns = df.columns.str.capitalize()
macd_impl(df)
macd_features(df)
volume_features(df)
price_features(df)
indicators_features(df)
ema_features(df)
get_ribbon_features(df)
df.dropna(inplace = True)

/tmp/ipykernel_465/487214107.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['1h_ema'] = df['Close'].resample('1H').mean().reindex(df.index, method='ffill')


In [39]:
class trade_status_strategy(Strategy):
    
    def init(self):
        #empty function in which we should put internal calcs for indicators, required even if empty
        pass

    def next(self):

        price = self.data.Close[-1]
        current_atr = self.data.atr[-1]
        trade_signal = self.data.Trade_Direction[-1]

        if not self.position:
            if trade_signal == -1:  # Short Signal
                # Calculate levels based on the moment of the signal
                sl_price = price + (current_atr * 2.5)
                tp_price = price - (current_atr * 5)
                
                # Execute at current close (or next open) using these static values
                self.sell(size=0.01, sl=sl_price, tp=tp_price)

            elif trade_signal == 1:  # Long Signal
                sl_price = price - (current_atr * 2.5)
                tp_price = price + (current_atr * 5)
                
                self.buy(size=0.01, sl=sl_price, tp=tp_price)

In [40]:
bt = Backtest(
    df,
    trade_status_strategy,
    cash=100000000,
    exclusive_orders=True,
    trade_on_close=True
)
results = bt.run()
results

/tmp/ipykernel_465/1750268398.py:1: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt = Backtest(
/tmp/ipykernel_465/1750268398.py:8: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results = bt.run()


Start                     2022-12-10 08:30...
End                       2025-12-08 11:40...
Duration                   1094 days 03:10:00
Exposure Time [%]                    92.17281
Equity Final [$]              101539497.44789
Equity Peak [$]               101634813.40294
Return [%]                             1.5395
Buy & Hold Return [%]               436.18103
Return (Ann.) [%]                     0.50794
Volatility (Ann.) [%]                 0.37157
CAGR [%]                              0.51096
Sharpe Ratio                          1.36701
Sortino Ratio                         2.07976
Calmar Ratio                          1.57657
Alpha [%]                              1.0412
Beta                                  0.00114
Max. Drawdown [%]                    -0.32218
Avg. Drawdown [%]                    -0.00943
Max. Drawdown Duration      198 days 21:40:00
Avg. Drawdown Duration        1 days 12:07:00
# Trades                                 5534
Win Rate [%]                      

In [41]:
df_results = results['_trades']
implement_trades(df, df_results)
df_trades = df.loc[(df['Trades'] == 0) | (df['Trades'] == 1)]

In [42]:
df.to_csv('btc_3y_with_features_04.csv')
df_trades.to_csv('btc_trades_features_04.csv')